# 06 — Species-level Contributions to GMMs

**Paper title:** Maternal gut microbiome diversity and functional potential in early pregnancy are associated with large-for-gestational-age birth (SweMaMi cohort)

**Purpose:** Extracts KO gene family contributions from HUMAnN2 output, stratified by species, to identify which taxa drive the AGA-associated and LGA assocaited modules at TP1.

**Produces:** Figure 9 (KO contributions of Anaerostipes hadrus)


## Setup


In [ ]:
library(dplyr)
library(data.table)
library(tibble)
library(ggplot2)
library(tidyverse)
library(ggpubr)
library(rstatix)
library(ggtext)

In [ ]:
# Define base path 
base_path <- "data"




## 1. Load stratified KO abundances (HUMAnN2)


In [ ]:
##GMM definition file: 
gmm_raw <- readLines(file.path(base_path,"GMM_definition.txt"))
head(gmm_raw)

In [ ]:
kegg_path <- "data_2"

In [ ]:
# Get all KEGG files, output generated by HUMAnN
kegg_files <- list.files(kegg_path,
                          pattern = "humann_kegg.tsv",
                          full.names = TRUE)
kegg_files

In [ ]:
cat("Number of KEGG files:", length(kegg_files), "\n")

In [ ]:
meta_data_tp1 <- read.csv(file.path(base_path, "meta_data_tp1.csv"), 
                           header = TRUE)
head(meta_data_tp1)
dim(meta_data_tp1)

In [ ]:
# PASS 1 — collect module-level info (hier, ref, pos, cpd)
module_list <- list()

current_gmm        <- NULL
current_name       <- NULL
current_pos        <- NA
current_cpd_input  <- NA
current_cpd_output <- NA
current_hier       <- NULL
current_ref        <- NULL

for (line in gmm_raw) {
  
  if (grepl("^MF", line)) {
    parts          <- strsplit(line, "\t")[[1]]
    current_gmm    <- parts[1]
    current_name   <- parts[2]
    current_pos    <- NA
    current_cpd_input  <- NA
    current_cpd_output <- NA
    current_hier   <- NULL
    current_ref    <- NULL
    
  } else if (grepl("^cpd", line)) {
    parts              <- strsplit(line, "\t")[[1]]
    current_cpd_input  <- gsub("\\[|\\]", "",
                                ifelse(length(parts) >= 2, parts[2], NA))
    current_cpd_output <- gsub("\\[|\\]", "",
                                ifelse(length(parts) >= 3, parts[3], NA))
    
  } else if (grepl("^pos", line)) {
    parts       <- strsplit(line, "\t")[[1]]
    current_pos <- ifelse(length(parts) >= 2, parts[2], NA)
    
  } else if (grepl("^hier", line)) {
    parts        <- strsplit(line, "\t")[[1]]
    current_hier <- parts[-1]
    current_hier <- current_hier[current_hier != ""]
    
  } else if (grepl("^ref", line)) {
    parts       <- strsplit(line, "\t")[[1]]
    current_ref <- parts[-1]
    current_ref <- current_ref[current_ref != ""]
    
  } else if (grepl("^///", line)) {
    module_list[[length(module_list) + 1]] <- data.frame(
      GMM           = current_gmm,
      GMM_name      = current_name,
      cpd_input     = current_cpd_input,
      cpd_output    = current_cpd_output,
      pos           = current_pos,
      hier          = ifelse(is.null(current_hier), NA,
                             paste(current_hier, collapse = " | ")),
      ref           = ifelse(is.null(current_ref), NA,
                             paste(current_ref, collapse = " | ")),
      stringsAsFactors = FALSE
    )
  }
}

gmm_module_table <- bind_rows(module_list)

# PASS 2 — collect KO level info
gmm_list    <- list()
current_gmm <- NULL
step_number <- 0

for (line in gmm_raw) {
  
  if (grepl("^MF", line)) {
    parts       <- strsplit(line, "\t")[[1]]
    current_gmm <- parts[1]
    step_number <- 0
    
  } else if (grepl("^K", line)) {
    ko_groups <- strsplit(line, "\t")[[1]]
    
    for (group in ko_groups) {
      step_number <- step_number + 1
      alt_kos     <- strsplit(group, ",")[[1]]
      alt_kos     <- trimws(alt_kos)
      alt_kos     <- alt_kos[grepl("^K\\d{5}", alt_kos)]
      
      for (ko in alt_kos) {
        gmm_list[[length(gmm_list) + 1]] <- data.frame(
          GMM            = current_gmm,
          KO             = ko,
          step           = step_number,
          is_alternative = length(alt_kos) > 1,
          stringsAsFactors = FALSE
        )
      }
    }
    
  } else if (grepl("^///", line)) {
    step_number <- 0  
  }
}

gmm_ko_table <- bind_rows(gmm_list)

# JOIN module info to KO table
gmm_ko_table <- gmm_ko_table %>%
  left_join(gmm_module_table, by = "GMM")

# Add total steps per module
total_steps <- gmm_ko_table %>%
  group_by(GMM) %>%
  summarise(total_steps = max(step), .groups = "drop")

GMM_def <- gmm_ko_table %>%
  left_join(total_steps, by = "GMM")

In [ ]:
write_tsv(GMM_def,
          file.path(base_path, "gmm_ko_table.tsv"))

In [ ]:
# Read and merge all KEGG files generated by HUMAnN

kegg_merged <- NULL

for (i in 1:length(kegg_files)) {
  
  tmp <- fread(kegg_files[i], sep = "\t")
  
  if (is.null(kegg_merged)) {
    kegg_merged <- tmp
  } else {
    kegg_merged <- merge(kegg_merged, tmp,
                         by = colnames(tmp)[1],
                         all = TRUE)
  }
  
  if (i %% 5 == 0) {
    message(paste("Files merged so far:", i))
  }
  
  rm(tmp)
  gc()
}


cat("Dimensions:", dim(kegg_merged), "\n")


In [ ]:
# Rename first column
colnames(kegg_merged)[1] <- "GeneFamily"

# Replace NAs with 0
kegg_merged[is.na(kegg_merged)] <- 0

# Parse KO and species
kegg_clean <- kegg_merged %>%
  as_tibble() %>%
  filter(grepl("\\|", GeneFamily),             
         !grepl("unclassified", GeneFamily)) %>% 
  mutate(
    KO      = gsub("\\|.*", "", GeneFamily),
    species = gsub(".*s__", "", GeneFamily),
    species = gsub("\\.t__.*", "", species),
    species = gsub("_", " ", species)
  )


In [ ]:
fetch <- colnames(kegg_merged %>% select(-c(GeneFamily)))
extracted_ids <- gsub(".*__(\\d+)__.*", "\\1", fetch)
length(extracted_ids)
colnames(kegg_merged) <- c("GeneFamily",  extracted_ids)
gene_fam <- colnames(kegg_merged)
kegg_merged_1 <- kegg_merged

In [ ]:
write_tsv(kegg_merged_1, 
          file.path(base_path, "unfiltered_kegg_sp_all_data.tsv"))

In [ ]:
length(intersect(colnames(kegg_merged_1), meta_data_tp1$kit1.faecal_sample.barcode))

In [ ]:
kegg_merged_2 <- as.data.frame(kegg_merged_1)[,
  c(1, which(colnames(kegg_merged_1) %in% meta_data_tp1$kit1.faecal_sample.barcode))
]

# Rename first column for easy handling
colnames(kegg_merged_2)[1] <- "feature"

head(kegg_merged_2)
dim(kegg_merged_2)


## 2. Map KOs to target GMMs

In [ ]:
# Check what Row 3 looks like (UNGROUPED + unclassified + species)
kegg_sp_clean_1 <- kegg_merged_2 %>%
  as_tibble() %>%
  filter(grepl("\\|", feature),
         !grepl("UNMAPPED", feature)) %>%
  mutate(
    # KO — everything before
    KO = gsub("\\|.*", "", feature),
    
    # Extract genus part (between g__ and .s__)
    genus = case_when(
      grepl("g__", feature) ~
        gsub(".*g__(.*)\\.s__.*", "\\1", feature) %>%
        gsub("_", " ", .),
      TRUE ~ NA_character_
    ),
    
    # Extract species part (after s__)
    species = case_when(
      grepl("s__", feature) ~
        gsub(".*s__", "", feature) %>%
        gsub("\\.t__.*", "", .) %>%
        gsub("_", " ", .),
      TRUE ~ "unclassified"
    ),
    
    # Flag unclassified at GENUS level
    genus_classified = case_when(
      !grepl("g__", feature)                    ~ FALSE,
      grepl("unclassified", 
             gsub("\\.s__.*", "", feature))      ~ FALSE,  
      TRUE                                          ~ TRUE
    ),
    
    # Flag unclassified at SPECIES level
    species_classified = case_when(
      grepl("\\|unclassified$", feature)        ~ FALSE,  
      grepl("s__.*unclassified", feature)        ~ FALSE,  
      !grepl("s__", feature)                     ~ FALSE,  
      TRUE                                          ~ TRUE
    ),
    
    # KO type
    ko_type = case_when(
      grepl("^K\\d{5}", feature)  ~ "KO",
      grepl("^UNGROUPED", feature) ~ "UNGROUPED",
      TRUE                            ~ "other"
    ),
    
    # Row type
    row_type = case_when(
      ko_type == "KO"       & species_classified  & genus_classified  ~ "KO_species_classified",
      ko_type == "KO"       & species_classified  & !genus_classified ~ "KO_species_unclassified_genus",
      ko_type == "KO"       & !species_classified & genus_classified  ~ "KO_unclassified_species",
      ko_type == "KO"       & !species_classified & !genus_classified ~ "KO_fully_unclassified",
      ko_type == "UNGROUPED"& species_classified  & genus_classified  ~ "UNGROUPED_species_classified",
      ko_type == "UNGROUPED"& species_classified  & !genus_classified ~ "UNGROUPED_species_unclassified_genus",
      ko_type == "UNGROUPED"& !species_classified                     ~ "UNGROUPED_unclassified",
      TRUE                                                            ~ "other"
    )
  )

# Verify
kegg_sp_clean_1 %>%
  count(row_type) %>%
  mutate(pct = round(n/sum(n)*100, 1)) %>%
  arrange(desc(n))

In [ ]:
# PRIMARY — KO and species both available (best for GMM linking)
main <- kegg_sp_clean_1 %>%
  filter(row_type %in% c("KO_species_classified",
                          "KO_species_unclassified_genus"))

# SECONDARY — species known but no KO
ungrouped <- kegg_sp_clean_1 %>%
  filter(row_type %in% c("UNGROUPED_species_classified",
                          "UNGROUPED_species_unclassified_genus"))

# DISCARD for GMM analysis — no species or no KO info
unclassified <- kegg_sp_clean_1 %>%
  filter(row_type %in% c("KO_fully_unclassified",
                          "UNGROUPED_unclassified"))

In [ ]:
write_tsv(main, 
          file.path(base_path, "kegg_KO_species_classified.tsv"))

write_tsv(ungrouped, 
          file.path(base_path, "kegg_UNGROUPED_species.tsv"))
write_tsv(unclassified, 
          file.path(base_path, "kegg_unclassified.tsv"))

write_tsv(kegg_sp_clean_1,
          file.path(base_path, "kegg_all_parsed.tsv"))


## 3. Summarise per-species contributions


In [ ]:

# From MaAsLin3 results
# Prevalent in LGA
lga_species <- c(
  "Oscillibacter sp ER4",
  "GGB9299 SGB14260",
  "GGB3146 SGB4159",
  "Dysosmobacter sp BX15",
  "Anaerotruncus rubiinfantis",
  "Oscillospiraceae unclassified SGB15256",
  "GGB9790 SGB15413",
  "GGB80011 SGB15267",
  "Candidatus Allochristensenella caecavium",
  "GGB2958 SGB3940",
  "GGB9765 SGB15382",
  "Clostridiales bacterium NSJ 40",
  "GGB9788 SGB15411"
)

# Prevalent/abundant in AGA
aga_species <- c(
  "Clostridium spiroforme",
  "Anaerostipes hadrus",
  "GGB3602 SGB4574"
)

all_sig_species <- c(lga_species, aga_species)

In [ ]:
# Define the modules of interest (common between the Maaslin3 and LR methods,here I considered the raw p value)
lga_modules <- c("MF0008","MF0040", "MF0032", "MF0034")
aga_modules  <- c("MF0102", "MF0089", "MF0073")
all_modules  <- c(lga_modules, aga_modules)


In [ ]:
gmm_ko_table_1 <- read_tsv(
  file.path(base_path, "gmm_ko_table"),
  show_col_types = FALSE
)

head(gmm_ko_table_1)

In [ ]:
# Get KOs for these modules
target_kos <- gmm_ko_table_1 %>%
  filter(GMM %in% all_modules) %>%
  select(GMM, GMM_name, KO, step, 
         is_alternative, total_steps) %>%
  mutate(
    group_association = ifelse(GMM %in% lga_modules,
                               "LGA", "AGA")
  )


In [ ]:
target_kos %>%
  count(GMM, GMM_name, group_association) %>%
  arrange(group_association)

In [ ]:
# Step 1 — Use exact matches for primary analysis
kegg_sig <- kegg_sp_clean_1 %>%
  filter(row_type == "KO_species_classified") %>%
  mutate(species = trimws(gsub("_", " ", species))) %>%
  filter(species %in% c("Anaerostipes hadrus",
                         "Clostridium spiroforme"))
head(kegg_sig)

In [ ]:
table(kegg_sig$species)

In [ ]:
# Step 2 — Connect to modules(common modules). Here i am looking for both species and modules. 
species_module <- kegg_sig %>%
  inner_join(
    target_kos %>% select(GMM, GMM_name, KO,
                           step, is_alternative,
                           total_steps, group_association),
    by = "KO"
  )
head(species_module)


In [ ]:
# Step 3 — Check what I found
species_module %>%
  count(species, GMM, GMM_name, group_association) %>%
  arrange(species, GMM)

In [ ]:
# Filter kegg for significant species, same as before. Here, I am using just the main table instead of the gmm_ko table
kegg_sig <- main %>%           # KO_species_classified table
  filter(species %in% all_sig_species)
head(kegg_sig)

In [ ]:
kegg_sig$species %>% unique() %>% sort()

In [ ]:
# Check if species names match between tables
# Names in kegg_sig
kegg_sig$species %>% unique() %>% sort()

# Compare with my sig_species list
setdiff(all_sig_species, 
        unique(kegg_sig$species))

In [ ]:
# See exact species names in kegg_sig
kegg_sig$species %>% 
  unique() %>% 
  sort() %>%
  print()


In [ ]:
kegg_sp_clean_1 %>%
  filter(species == "Clostridium spiroforme")

In [ ]:
#kegg_sp_clean_1

# What modules does Clostridium spiroforme contribute to?
kegg_sp_clean_1 %>%
  filter(species == "Clostridium spiroforme") %>%
  select(KO) %>%
  distinct() %>%
  inner_join(gmm_ko_table_1 %>% 
               select(KO, GMM, GMM_name),
             by = "KO") %>%
  count(GMM, GMM_name) %>%
  arrange(desc(n))

In [ ]:
species_module <- kegg_sig %>%
  inner_join(
    target_kos %>% select(GMM, GMM_name, KO, 
                           step, is_alternative, 
                           total_steps, group_association),
    by = "KO"
  )
head(species_module )

In [ ]:
# Check structure — should have sample columns
cat("Dimensions:", dim(species_module), "\n")
cat("Columns:", colnames(species_module), "\n")

# Verify sample columns are present
colnames(species_module) %>%
  .[grepl("^1000", .)] %>%   # sample IDs start with 1000
  head(5)

In [ ]:
# Identify non-sample columns to exclude from pivot
non_sample_cols <- c("feature", "KO", "genus", 
                     "species", "genus_classified",
                     "species_classified", "ko_type", 
                     "row_type", "GMM", "GMM_name",
                     "step", "is_alternative",
                     "total_steps", "group_association")

In [ ]:
# Check which of these exist in my table
non_sample_cols <- non_sample_cols[
  non_sample_cols %in% colnames(species_module)
]

In [ ]:
cat("Non-sample columns found:", non_sample_cols, "\n")

In [ ]:
meta_data_tp1$kit1.faecal_sample.barcode <- as.character(meta_data_tp1$kit1.faecal_sample.barcode)
class(meta_data_tp1$kit1.faecal_sample.barcode)

In [ ]:
# Pivot to long format
species_module_long <- species_module %>%
  pivot_longer(
    cols      = -all_of(non_sample_cols), 
    names_to  = "kit1.faecal_sample.barcode",
    values_to = "abundance"
  ) %>%
  left_join(
    meta_data_tp1 %>% select(kit1.faecal_sample.barcode	, Group),
    by = "kit1.faecal_sample.barcode"
  ) %>%
  filter(!is.na(Group))

In [ ]:
head(species_module_long)
dim(species_module_long)

In [ ]:
# Check join worked
cat("Samples matched:", 
    sum(!is.na(species_module_long$Group)), "\n")
cat("Samples not matched:", 
    sum(is.na(species_module_long$Group)), "\n")

In [ ]:
# Summary
species_module_summary <- species_module_long %>%
  group_by(species, GMM, GMM_name,
           group_association, Group) %>%
  summarise(
    mean_abundance = mean(abundance, na.rm = TRUE),
    sd_abundance   = sd(abundance,   na.rm = TRUE),
    .groups = "drop"
  )

species_module_summary

In [ ]:
# Which species connect to which modules?
species_module %>%
  count(species, GMM, GMM_name, group_association) %>%
  arrange(group_association, GMM, desc(n))

In [ ]:
# What % of module steps does each species cover?
completeness <- species_module %>%
  group_by(species, GMM, GMM_name, 
           group_association, total_steps) %>%
  summarise(
    steps_covered  = n_distinct(step),
    KOs_found      = n_distinct(KO),
    .groups = "drop"
  ) %>%
  mutate(
    completeness_pct = round(steps_covered / 
                               total_steps * 100, 1)
  ) %>%
  arrange(group_association, GMM, 
          desc(completeness_pct))
completeness

In [ ]:
# Test if difference is significant
species_module_long %>%
  filter(species == "Anaerostipes hadrus",
         GMM %in% c("MF0102", "MF0089", "MF0073")) %>%
  group_by(GMM, GMM_name) %>%
  summarise(
    wilcox_p = wilcox.test(
      abundance[Group == "Control"],
      abundance[Group == "Case"]
    )$p.value,
    mean_AGA = mean(abundance[Group == "Control"],
                    na.rm = TRUE),
    mean_LGA = mean(abundance[Group == "Case"],
                    na.rm = TRUE),
    fold_diff = mean_AGA / mean_LGA,
    .groups = "drop"
  ) %>%
  mutate(
    direction = ifelse(mean_AGA > mean_LGA,
                       "Higher in AGA",
                       "Higher in LGA"),
    fold_diff = round(fold_diff, 2),
    wilcox_p  = round(wilcox_p, 4)
  )


## 4. Plot Anaerostipes hadrus contributions (Figure 9)


In [ ]:
# Prepare data
plot_data <- species_module_long %>%
  filter(species == "Anaerostipes hadrus",
         GMM %in% c("MF0073", "MF0102", "MF0089")) %>%
  mutate(
    GMM_label = case_when(
      GMM == "MF0073" ~ "MF0073\nPyruvate:ferredoxin\noxidoreductase",
      GMM == "MF0102" ~ "MF0102\nsulfate\nreduction(dissimilatory)",
      GMM == "MF0089" ~ "MF0089\nButyrate\nproduction II"
    )
  )

In [ ]:
plot_data$Group <- factor(plot_data$Group,
                          labels = c("Control", "Case"))

In [ ]:
a_hardus_plot <- ggplot(plot_data, 
       aes(x    = Group, 
           y    = abundance, 
           fill = Group)) +
  geom_boxplot(outlier.shape = NA,
               alpha = 0.7,
               width = 0.5) +
  geom_jitter(aes(color = Group),
              width = 0.15,
              size  = 1.5,
              alpha = 0.5) +
  facet_wrap(~GMM_label, 
             nrow   = 1) +                      
  scale_fill_manual(values  = c("Control" = "#40E0D0",
                                 "Case" = "deeppink")) +
  scale_color_manual(values = c("Control" = "#40E0D0",
                                 "Case" = "deeppink")) +
  stat_compare_means(
    method       = "wilcox.test",
    label        = "p.format",
    label.x      = 1.5,
    label.y      = 33,                            
    size         = 4
  ) +
  coord_cartesian(ylim = c(0, 35)) +             
  theme_minimal(base_size = 12) +
  theme(
    legend.position  = "none",
    strip.text       = element_text(size = 10,
                                     face = "bold"),
    axis.title       = element_text(size = 11),
    panel.grid.minor = element_blank()
  ) +
  labs(
    title = "*Anaerostipes hadrus* contribution to GMM",
    x     = "Birth weight group",
    y     = "KO abundance"
  ) +
  theme(
    plot.title = element_markdown()
  )
a_hardus_plot

In [ ]:
ggsave(
  file.path(base_path, "figure_9.pdf"),
    width     = 460,    
       height    = 460,
       units     = "mm",
       limitsize = FALSE,
       device    = cairo_pdf)